# Plant Disease Classifier — Single Epoch Training
**Model:** EfficientNet-B0 (pretrained)  
**Dataset:** `dataset/train/`, `dataset/val/`  
**Saves to:** `plant_model.pt`

In [ ]:
import subprocess, sys
for pkg in ['torch','torchvision','opencv-python','joblib','scikit-learn','Pillow']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages ready')

In [ ]:
import random
from pathlib import Path
import cv2, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# ── Configuration ─────────────────────────────────
DATA_DIR   = Path('dataset')
MODEL_PATH = Path('plant_model.pt')
EPOCHS     = 1          # single epoch
BATCH_SIZE = 32
IMAGE_SIZE = 224
LR         = 3e-4
WORKERS    = 2
SEED       = 42
METHODS    = ['none', 'clahe', 'gamma_bright', 'gamma_dark', 'hist_eq']
print('Config OK')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def enhance(rgb, method):
    if method == 'none':
        return rgb.copy()
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    lum = lab[:, :, 0]
    if method == 'clahe':
        lum = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lum)
    elif method in {'gamma_bright', 'gamma_dark'}:
        g = 0.65 if method == 'gamma_bright' else 1.45
        table = np.array([((i / 255.0) ** g) * 255 for i in range(256)]).astype('uint8')
        lum = cv2.LUT(lum, table)
    elif method == 'hist_eq':
        lum = cv2.equalizeHist(lum)
    lab[:, :, 0] = lum
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

class RandomEnhancement:
    def __call__(self, img):
        rgb = np.asarray(img.convert('RGB'))
        return Image.fromarray(enhance(rgb, random.choice(METHODS)))

def build_transforms(size=224):
    norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    train_tf = transforms.Compose([
        RandomEnhancement(),
        transforms.Resize((size, size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(12),
        transforms.ToTensor(),
        norm,
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        norm,
    ])
    return train_tf, eval_tf

print('Helpers defined')

In [ ]:
set_seed(SEED)
train_tf, eval_tf = build_transforms(IMAGE_SIZE)
train_data = datasets.ImageFolder(DATA_DIR / 'train', transform=train_tf)
val_data   = datasets.ImageFolder(DATA_DIR / 'val',   transform=eval_tf)
train_loader = DataLoader(train_data, BATCH_SIZE, shuffle=True,  num_workers=WORKERS)
val_loader   = DataLoader(val_data,   BATCH_SIZE, shuffle=False, num_workers=WORKERS)
print('Classes :', len(train_data.classes))
print('Train   :', len(train_data), 'images')
print('Val     :', len(val_data), 'images')
print(train_data.classes)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
weights = models.EfficientNet_B0_Weights.DEFAULT
model   = models.efficientnet_b0(weights=weights)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(train_data.classes))
model   = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
loss_fn   = nn.CrossEntropyLoss()
print('Model ready')

In [ ]:
def run_epoch(model, loader, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss = correct = total = 0
    for i, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad()
        logits = model(images)
        loss   = loss_fn(logits, labels)
        if training:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
        if (i + 1) % 10 == 0:
            print(f'  batch {i+1}/{len(loader)}  loss={total_loss/total:.4f}  acc={correct/total:.4f}')
    return total_loss / total, correct / total

print('=' * 50)
print('  EPOCH 1 / 1  --  TRAINING')
print('=' * 50)
train_loss, train_acc = run_epoch(model, train_loader, device, optimizer)
print(f'Train  loss={train_loss:.4f}  acc={train_acc:.4f}')

In [ ]:
print('Running validation...')
with torch.no_grad():
    val_loss, val_acc = run_epoch(model, val_loader, device, optimizer=None)
print(f'Val    loss={val_loss:.4f}  acc={val_acc:.4f}')

In [ ]:
checkpoint = {'state_dict': model.state_dict(), 'classes': train_data.classes}
torch.save(checkpoint, MODEL_PATH)
print('Model saved to:', MODEL_PATH.resolve())
print('Train acc:', f'{train_acc:.2%}')
print('Val   acc:', f'{val_acc:.2%}')

In [ ]:
print('=' * 40)
print('        TRAINING SUMMARY')
print('=' * 40)
print('  Epochs     :', 1)
print('  Batch size :', BATCH_SIZE)
print('  LR         :', LR)
print('  Device     :', device)
print('  Classes    :', len(train_data.classes))
print('-' * 40)
print('  Train Loss :', f'{train_loss:.4f}')
print('  Train Acc  :', f'{train_acc:.2%}')
print('  Val Loss   :', f'{val_loss:.4f}')
print('  Val Acc    :', f'{val_acc:.2%}')
print('=' * 40)